# YuE2 を Kaggle の無料 GPU で動かす（T4 x2 / 16 GB 向け調整済み）

[YuE2](https://github.com/multimodal-art-projection/YuE)（`m-a-p/YuE2-3B` + `YuE2-Vae`）は
公式には **BF16 対応 GPU / 24 GB VRAM** を前提にしています。Kaggle の無料枠は
**P100 (16 GB, cc 6.0)** または **T4 x2 (各 16 GB, cc 7.5)** で、セッションは最大 12 時間、
GPU は週 30 時間まで、`/kaggle/working` は 20 GB まで保存されます。

このノートブックは、その差を埋めるための最小限の調整だけを行います。

| 項目 | 対応 |
|---|---|
| 16 GB VRAM | `memory_budget_gib` を実測 VRAM から算出、`vae_core_frames=512`、VAD は常に 512 フレーム単位 |
| BF16 非ネイティブ (T4) | PyTorch が Turing では BF16 をエミュレートするため**実行は可能**。ただし遅い（実測で確認） |
| P100 (cc 6.0) | `torch.cuda.is_bf16_supported()` が False になり pipeline が起動を拒否 → **T4 x2 を選び直してください** |
| セッション 12 時間 / 20 分アイドル | 生成を **plan → semantic → flow matching → decode** の4段に分割し、各段をディスクに保存。時間切れでも**再開可能** |
| どれくらい時間がかかるか不明 | 実モデルで 1〜2 分の**実測ベンチ**を取り、曲ごとの所要時間(ETA)を先に出します |

**モデルの重みは CC BY-NC 4.0（非商用）です。**個人クリエイターの制作物の利用は許諾されていますが、
企業の商用利用には別途ライセンスが必要です（[MODEL_LICENSE](https://github.com/multimodal-art-projection/YuE/blob/main/MODEL_LICENSE)）。
カグルのノートは既定で公開になるため、出力を公開したくない場合は右上の Settings で Private にしてください。

## 使い方

1. この `.ipynb` を Kaggle の **Code → New Notebook → File → Import Notebook** で取り込みます。
2. 右上 **Settings** で **Accelerator = GPU T4 x2**、**Internet = On** にします。
3. 下の「設定」セルを書き換えて、**Save & Run All (Commit)** または上から順に実行します。
   - 長時間かかる曲は **Commit** 実行（最大12時間、ブラウザを閉じても継続）を推奨。対話実行は20分無操作で停止します。
   - 途中で止まっても、同じ設定でもう一度実行すれば続きから再開します。
4. 出力は `/kaggle/working/outputs/<id>/`（`audio.flac` など）に保存され、ノートブックの Output からダウンロードできます。

## 1. 設定

In [ ]:
# ============================ 設定（ここだけ編集） ============================
STYLE = (
    "English, warm piano pop, expressive female voice, acoustic piano, "
    "rounded bass and light drums, lyrical memorable melody, unhurried phrasing, 88 BPM"
)
LYRICS = """[Verse]
Neon fades along the lane
Footsteps keep the time of rain
[Chorus]
Let the day come into view
Every road begins with you"""

COT = "full"            # full（メロディ+コード譜を自動生成）/ melody / off（譜面なし）
SECONDS = 120.0         # 生成する長さ（秒）。まずは 60〜120 秒で試すのがおすすめ
SEED = 831001
ODE_STEPS = 32          # 公式デフォルト。下げると速いが品質はトレードオフ（8〜16 で実験）
PLAN_MAX_TOKENS = None  # ABC 譜生成の上限（公式は 4096）。スモークテスト時のみ 512〜1024 などに
                        # 下げる。譜面が途中で切れる=品質が変わるので本番比較には使わないこと
ABC = None              # 手持ちの ABC 譜を使う場合は文字列で指定（例: open("score.abc").read()）
                        # ABC を渡すと譜面生成(plan)を丸ごと省略でき、その分速くなります

OUTPUT_ID = "song-01"
BUDGET_MINUTES = 600    # この実行で使う時間の上限（12h セッションなら 660 くらいまで可）
SAFETY_MINUTES = 10     # セッション破棄に備えて残す余裕

RUN_BENCHMARK = True    # 実モデルで実測ベンチを取る（1〜2分）。ETA を出してから本番へ
FORCE_RUN = False       # ETA が予算を超えていても強行する

INSTALL_MODE = "image"  # "image"(既定): 同梱の numpy/torch を保つ（安全・推奨）
                        # "pinned": 公式ピン留め。numpy 2.2.6 を入れるため scipy/scikit-learn と
                        #           衝突しうる（壊れたら Restart session して "image" に戻す）
PERSIST_MODEL = False   # True にすると HF キャッシュを /kaggle/working に置く（後で Dataset 化しやすい）
REPO_URL = "https://github.com/multimodal-art-projection/YuE.git"  # 自分の fork に差し替えてもOK
REPO_REF = "main"       # fork の branch 名（例: "arena/01a0f78a-yue"）

import os
from pathlib import Path

WORKDIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
REPO_DIR = WORKDIR / "YuE"
OUTDIR = WORKDIR / "outputs" / OUTPUT_ID
if PERSIST_MODEL:
    os.environ["HF_HOME"] = str(WORKDIR / "hf-cache")
print("output directory:", OUTDIR)

## 2. GPU の確認（インストール前）

In [ ]:
# --- まず torch 抜きで確認: GPU の種類とインターネット接続 ---
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!python -c "import sys; print('python', sys.version.split()[0])"
!df -h /kaggle/working | tail -1

# インターネット: git clone / pip / モデル(約7.8GB)のDL に必須です
import socket

def reachable(host, timeout=5):
    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError as exc:
        print(f"  {host}: NG ({exc})")
        return False

INTERNET = all([reachable("github.com"), reachable("pypi.org"), reachable("huggingface.co")])
if INTERNET:
    print("internet: OK (github / pypi / huggingface に到達できます)")
else:
    print()
    print(">>> インターネットに接続できていません。右の Settings で Internet を [On] にしてください。")
    print(">>> On にしても直らないときは Run -> Restart session してから、上から順に実行し直します。")
    print(">>> 電話番号認証が未完了だと Internet も GPU も選べません（Settings -> Phone Verification）。")

## 3. 依存のインストール

In [ ]:
# --- リポジトリの取得と依存のインストール（初回 3〜8 分程度） ---
import subprocess, sys, os, importlib, importlib.metadata as md

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

def version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

def major_minor(text):
    try:
        parts = str(text).split("+")[0].split(".")
        return int(parts[0]), int(parts[1])
    except Exception:
        return (0, 0)

print("image before:", {n: version(n) for n in
                        ("torch", "numpy", "scipy", "scikit-learn", "transformers")})

try:
    if not (REPO_DIR / "pyproject.toml").is_file():
        if not INTERNET:
            raise RuntimeError("Internet が Off です（セル2の診断を参照）")
        if REPO_DIR.exists() and not any(REPO_DIR.iterdir()):
            REPO_DIR.rmdir()          # 失敗した clone の空ディレクトリが残っていたら片付ける
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
                       check=True)

    if INSTALL_MODE == "pinned":
        # 公式レシピそのまま。ただし numpy==2.2.6 を入れるため、イメージ同梱の scipy /
        # scikit-learn と不整合になることがあります（numpy._core.umath の ImportError）。
        print("WARNING: pinned モードは numpy を差し替えます。壊れたら Restart session して")
        print("         INSTALL_MODE='image'（既定）でやり直してください。")
        pip(str(REPO_DIR))
    else:
        # 既定: イメージ同梱の numpy / torch は触らない（numpy を下げると scipy/sklearn が壊れる）
        pip("--no-deps", str(REPO_DIR))
        pip("transformers==4.57.6", "huggingface-hub==0.36.2", "safetensors==0.7.0",
            "tiktoken==0.12.0", "soundfile==0.13.1", "accelerate==1.13.0")
        torch_version = version("torch")
        if major_minor(torch_version or "0") < (2, 10):
            print(f"image torch={torch_version}: 上流は torch 2.10 前提なので入れ替えます")
            pip("torch==2.10.0")
        elif not str(torch_version).startswith("2.10"):
            print(f"note: image torch is {torch_version}; 上流のピンは 2.10.0 です（そのまま使います）")
except subprocess.CalledProcessError as exc:
    print("コマンドが失敗しました:", exc.cmd)
    print()
    print("確認すること:")
    print("  1) Settings -> Internet が [On] か（Off だと 'Could not resolve host' で失敗します）")
    print("  2) Settings -> Accelerator が [GPU T4 x2] か（P100 では YuE2 は動きません）")
    print("  3) 一時的なネットワーク障害なら、1分ほど待ってこのセルを再実行")
    print("  4) /kaggle/working の空き容量（保存できるのは 20GB まで）")
    raise

# --- 整合性チェック（numpy を差し替えて scipy/sklearn が壊れていないか） ---
broken = []
for name in ("numpy", "scipy", "sklearn", "transformers", "torch", "soundfile", "tiktoken"):
    try:
        module = importlib.import_module(name)
        print(f"  OK  {name} {getattr(module, '__version__', '')}")
    except Exception as exc:
        broken.append(f"{name}: {type(exc).__name__}: {exc}")
        print(f"  NG  {name}: {type(exc).__name__}: {exc}")
try:
    from transformers import GenerationMixin  # noqa: F401  ← yue2 が実際に通る import 経路
    print("  OK  transformers.GenerationMixin（yue2 が必要とする import 経路）")
except Exception as exc:
    broken.append(f"transformers.generation: {type(exc).__name__}: {exc}")
    print(f"  NG  transformers.GenerationMixin: {type(exc).__name__}: {exc}")

if broken:
    print()
    print(">>> Python 環境が不整合です（典型: numpy を差し替えて scipy / scikit-learn が壊れた）。")
    print(">>> 対処: Run -> Restart session してから、INSTALL_MODE='image' で上から実行し直します。")
    raise RuntimeError("inconsistent environment: " + " | ".join(broken))

import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print(">>> GPU が見えていません。Settings -> Accelerator を GPU T4 x2 にして Restart session してください。")

## 4. 実行ドライバの書き出し

In [ ]:
%%writefile /kaggle/working/yue2_kaggle.py
#!/usr/bin/env python3
"""YuE2 on a free-tier Kaggle notebook: preflight, measured ETA, staged generation.

This module is a thin runner around the *unmodified* upstream ``yue2`` package.
It adds the parts that a 16 GB, non-native-BF16, 12-hour-session environment
needs:

* **Preflight.** Reports device, compute capability, VRAM, RAM, disk, and the
  pipeline's own BF16 gate, then derives safe ``YuE2Pipeline`` settings for this
  device instead of assuming the published 24 GB / native-BF16 baseline.
* **Honest estimates.** A one-to-two minute micro-benchmark of the real model
  turns the "24 GB BF16 GPU" guidance into a concrete per-song ETA for *this*
  GPU, so a run is never started blind.
* **Staged generation with resume.** plan -> semantic -> flow matching ->
  decode. Every stage is cached on disk, so a Kaggle session limit, a 20-minute
  interactive idle timeout, or a crashed kernel never throws away completed
  work: re-run the notebook and it continues.
* **Safety margins.** A wall-clock deadline is passed to the pipeline's
  ``cancelled`` hook so a stage stops cleanly before the session dies.

Nothing here changes model math, sampling defaults, or the artifact protocol.
The stages call ``YuE2Pipeline.plan / generate_semantic / synthesize / decode``
with the release configuration. Two clearly marked helpers reach for the
module-level functions the pipeline itself calls (``yue2.nar``) so flow matching
can checkpoint one chunk at a time; each has a public-API fallback.

Usage inside a Kaggle notebook (after installing the repository)::

    import yue2_kaggle as yk
    env = yk.environment(); yk.print_report(env)
    opts = yk.Options(outdir=Path("/kaggle/working/outputs/demo"), seconds=120)
    with yk.open_pipeline(env, opts) as handle:
        rates = yk.measure(handle)
        yk.print_estimate(yk.predict(rates, opts), opts)
        yk.run(handle, opts, rates=rates)
"""

from __future__ import annotations

import argparse
import json
import math
import os
import shutil
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

# ---------------------------------------------------------------------------
# Release constants (checked against the repository's assets/release.json).
# ---------------------------------------------------------------------------

FRAME_SECONDS = 1920 / 48000          # YuE2-Vae downsampling_ratio 1920 @ 48 kHz -> 25 fps
CONTEXT = 24576                       # YuE2 protocol context length
KV_BYTES_PER_TOKEN = 2 * 28 * 8 * 128 * 2      # k+v * layers * kv_heads * head_dim * bf16
MOT_WEIGHT_BYTES = 7_261_441_640      # m-a-p/YuE2-3B model.safetensors, bf16
VAE_WEIGHT_BYTES = 530_512_720        # m-a-p/YuE2-Vae model.safetensors, fp32
DEFAULT_MODEL = "m-a-p/YuE2-3B"
DEFAULT_VAE = "m-a-p/YuE2-Vae"

# Runtime errors that mean "this GPU cannot use the CUDA-graph/flash fast path".
GRAPH_FALLBACK_HINTS = (
    "flash", "flashattention", "sm_80", "ampere", "compute capability",
    "cuda graph", "cudagraph",
)

DEFAULT_STYLE = (
    "English, warm piano pop, expressive female voice, acoustic piano, "
    "rounded bass and light drums, lyrical memorable melody, unhurried phrasing, 88 BPM"
)
DEFAULT_LYRICS = (
    "[Verse]\nNeon fades along the lane\nFootsteps keep the time of rain\n"
    "[Chorus]\nLet the day come into view\nEvery road begins with you"
)


def frames_for_seconds(seconds: float) -> int:
    return max(1, int(math.ceil(float(seconds) / FRAME_SECONDS)))


def seconds_for_frames(frames: int) -> float:
    return float(frames) * FRAME_SECONDS


def hms(seconds: float | None) -> str:
    if seconds is None or not isinstance(seconds, (int, float)) or not math.isfinite(seconds):
        return "n/a"
    seconds = max(0.0, float(seconds))
    if seconds < 90:
        return f"{seconds:.0f}s"
    if seconds < 5400:
        return f"{seconds / 60:.1f} min"
    return f"{seconds / 3600:.2f} h"


# ---------------------------------------------------------------------------
# Preflight
# ---------------------------------------------------------------------------


def find_local_models(search_roots=(Path("/kaggle/input"),)) -> tuple[str | None, str | None]:
    """Locate a YuE2 model/VAE already attached as a Kaggle dataset (offline use).

    Only three directory levels are probed so a large attached dataset cannot
    turn the preflight into a filesystem crawl.
    """
    model = vae = None
    for root in search_roots:
        if not root.is_dir():
            continue
        candidates = []
        for depth in (1, 2, 3):
            candidates.extend(sorted(root.glob("/".join(["*"] * depth) + "/config.json")))
        for config_path in candidates:
            directory = config_path.parent
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
            except (OSError, ValueError):
                continue
            kind = config.get("model_type")
            if kind == "yue2" and (directory / "qwen.tiktoken").is_file() and model is None:
                model = str(directory)
            elif kind == "yue2_vae" and (directory / "model.safetensors").is_file() and vae is None:
                vae = str(directory)
    return model, vae


def _disk(path: Path) -> dict:
    try:
        usage = shutil.disk_usage(path)
        return {"path": str(path), "free_gib": usage.free / 2**30, "total_gib": usage.total / 2**30}
    except OSError as exc:
        return {"path": str(path), "error": str(exc)}


def _ram_gib() -> float | None:
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    except (ValueError, OSError, AttributeError):
        return None


def _reachable(host: str, timeout: float = 4.0) -> bool:
    """TCP probe; a clone/pip failure saying 'Could not resolve host' usually
    means the notebook's Internet switch is off, not a broken network."""
    import socket

    try:
        socket.create_connection((host, 443), timeout=timeout).close()
        return True
    except OSError:
        return False


def _matmul_probe(torch, device, size=2048, repeats=3) -> dict:
    """Effective BF16/FP16/FP32 GEMM throughput; exposes emulated-BF16 penalties."""
    rates: dict = {}
    for name, dtype in (("bf16", torch.bfloat16), ("fp16", torch.float16), ("fp32", torch.float32)):
        try:
            left = torch.randn(size, size, device=device, dtype=dtype)
            right = torch.randn(size, size, device=device, dtype=dtype)
            for _ in range(2):
                left @ right
            torch.cuda.synchronize(device)
            start = time.perf_counter()
            for _ in range(repeats):
                left @ right
            torch.cuda.synchronize(device)
            rates[name] = 2 * size**3 / ((time.perf_counter() - start) / repeats) / 1e12
            del left, right
        except Exception as exc:                     # pragma: no cover - device dependent
            rates[name] = None
            rates[f"{name}_error"] = f"{type(exc).__name__}: {exc}"
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return rates


def _import_probe() -> dict:
    """Import the packages yue2 pulls in, so a broken numpy/scipy mixture is
    reported at preflight instead of as a mid-generation ImportError.

    Installing the pinned recipe can downgrade numpy under an image-provided
    scipy/scikit-learn; the symptom is
    ``ImportError: cannot import name '_center' from 'numpy._core.umath'``.
    """
    import importlib

    results: dict = {}
    for name in ("numpy", "scipy", "sklearn", "transformers", "torch", "soundfile", "tiktoken"):
        try:
            module = importlib.import_module(name)
            results[name] = str(getattr(module, "__version__", "unknown"))
        except Exception as exc:
            results[name] = f"FAILED: {type(exc).__name__}: {exc}"
    try:
        from transformers import GenerationMixin       # noqa: F401 - the chain yue2 uses
        results["transformers.GenerationMixin"] = "OK"
    except Exception as exc:
        results["transformers.GenerationMixin"] = f"FAILED: {type(exc).__name__}: {exc}"
    return results


def environment(probe: bool = True) -> dict:
    """Collect everything needed to decide whether and how a run can proceed."""
    import torch

    report: dict = {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "on_kaggle": Path("/kaggle").exists(),
        "ram_gib": _ram_gib(),
        "devices": [],
        "matmul_tflops": {},
    }
    if report["cuda_available"]:
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            free, total = torch.cuda.mem_get_info(index)
            report["devices"].append({
                "index": index,
                "name": props.name,
                "memory_gib": total / 2**30,
                "free_gib": free / 2**30,
                "compute_capability": [props.major, props.minor],
                "multi_processor_count": props.multi_processor_count,
            })
        current = torch.cuda.current_device()
        report["current_device"] = f"cuda:{current}"
        try:
            report["bf16_supported"] = bool(torch.cuda.is_bf16_supported())
        except Exception as exc:                     # pragma: no cover - device dependent
            report["bf16_supported"] = None
            report["bf16_error"] = f"{type(exc).__name__}: {exc}"
        try:
            report["bf16_native"] = bool(torch.cuda.is_bf16_supported(including_emulation=False))
        except TypeError:                             # older torch without that argument
            report["bf16_native"] = None
        if probe:
            report["matmul_tflops"] = _matmul_probe(torch, torch.device(report["current_device"]))
    report["disk"] = {"working (/kaggle/working)": _disk(Path("/kaggle/working")),
                      "home cache (~/.cache)": _disk(Path.home()),
                      "root (/)": _disk(Path("/"))}
    report["local_model_dir"], report["local_vae_dir"] = find_local_models()
    report["internet"] = {"huggingface.co": _reachable("huggingface.co")}
    report["runtime_imports"] = _import_probe()
    return report


def verdict(report: dict) -> dict:
    """Turn the environment report into a pass/fail plus concrete advice."""
    checks: list[str] = []
    warnings: list[str] = []
    if not report.get("cuda_available"):
        checks.append("No CUDA GPU. Kaggle: Settings -> Accelerator -> 'GPU T4 x2', then re-run.")
        return {"ok": False, "checks": checks, "warnings": warnings}
    index = int(str(report.get("current_device", "cuda:0")).split(":")[-1])
    device = report["devices"][index]
    report["device"] = device
    major, minor = device["compute_capability"]
    checks.append(f"{device['name']} (cc {major}.{minor}, {device['memory_gib']:.1f} GiB, "
                  f"{device['free_gib']:.1f} GiB free), torch {report['torch']}")
    failed = {name: value for name, value in (report.get("runtime_imports") or {}).items()
              if isinstance(value, str) and value.startswith("FAILED")}
    if failed:
        checks.append("FAIL: the Python environment is inconsistent: "
                      + "; ".join(f"{name} -> {value}" for name, value in failed.items()))
        checks.append("Fix: Run -> Restart session, then re-run the install cell with "
                      "INSTALL_MODE='image' (never let pip downgrade numpy: the image's scipy and "
                      "scikit-learn are built against it).")
        return {"ok": False, "checks": checks, "warnings": warnings}
    if report.get("bf16_supported") is False:
        checks.append("FAIL: yue2's unquantized preset refuses this GPU because "
                      "torch.cuda.is_bf16_supported() is False. cc 6.0 (P100) is the usual "
                      "cause - switch the accelerator to 'GPU T4 x2'.")
        return {"ok": False, "checks": checks, "warnings": warnings}
    if report.get("bf16_native") is False:
        warnings.append("BF16 is emulated on this GPU (no native bf16 tensor cores): results are "
                        "correct, but expect several times the runtime of an Ampere-or-newer card.")
    if device["memory_gib"] < 15.0:
        warnings.append(f"{device['memory_gib']:.1f} GiB VRAM: keep songs short; offload_ar stays on.")
    if device["free_gib"] < device["memory_gib"] - 1.0:
        warnings.append(f"{device['free_gib']:.1f} GiB VRAM free of {device['memory_gib']:.1f} GiB "
                        "(another process holds the GPU); restart the session if that looks wrong.")
    if report.get("ram_gib") and report["ram_gib"] < 12:
        warnings.append(f"{report['ram_gib']:.0f} GiB system RAM: the 7.3 GB CPU copy of the model "
                        "plus the decoder may be tight.")
    working = report["disk"]["working (/kaggle/working)"]
    if "free_gib" in working and working["free_gib"] < 12:
        warnings.append(f"/kaggle/working has {working['free_gib']:.0f} GiB free; the ~7.8 GB of "
                        "model files plus artifacts need room (Kaggle persists at most 20 GB).")
    offline = (report.get("internet") or {}).get("huggingface.co") is False
    if offline and not report.get("local_model_dir"):
        checks.append("FAIL: huggingface.co is unreachable and no local model copy was found, so "
                      "the weights cannot be downloaded. Turn Settings -> Internet on, or attach "
                      "the models as a Kaggle dataset under /kaggle/input.")
        return {"ok": False, "checks": checks, "warnings": warnings}
    if offline:
        warnings.append("huggingface.co is unreachable; using the local model copy instead.")
    return {"ok": True, "checks": checks, "warnings": warnings}


def pipeline_settings(report: dict, *, ode_steps: int | None = None) -> dict:
    """Memory settings that fit the detected device instead of assuming 24 GB."""
    device = report.get("device") or report["devices"][0]
    vram = float(device["memory_gib"])
    settings = {
        "memory_budget_gib": max(4.0, min(24.0, vram - 0.5)),
        "offload_ar": vram < 15.5,
        "vae_core_frames": 512 if vram < 20 else 1024,
        "quantization": "none",          # FP8 needs cc >= 8.9; never available on the free tier
        "verify_hashes": True,
        "reserve_gib": max(0.0, vram - 2.0 - max(4.0, min(24.0, vram - 0.5))),
    }
    if ode_steps is not None:
        settings["ode_steps"] = int(ode_steps)
    return settings


def print_report(report: dict) -> dict:
    result = verdict(report)
    lines = ["=== YuE2 / Kaggle preflight ==="]
    lines += [f"  {line}" for line in result["checks"]]
    for name, value in (report.get("matmul_tflops") or {}).items():
        if isinstance(value, float):
            lines.append(f"  {name} GEMM: {value:.1f} TFLOP/s")
    if report.get("ram_gib"):
        lines.append(f"  system RAM: {report['ram_gib']:.1f} GiB")
    for label, disk in (report.get("disk") or {}).items():
        if "free_gib" in disk:
            lines.append(f"  disk {label}: {disk['free_gib']:.1f} GiB free")
    if report.get("local_model_dir"):
        lines.append(f"  local model (dataset) : {report['local_model_dir']}")
    if report.get("local_vae_dir"):
        lines.append(f"  local VAE (dataset)   : {report['local_vae_dir']}")
    runtime = report.get("runtime_imports") or {}
    if runtime:
        summary = ", ".join(f"{name} {value}" for name, value in runtime.items())
        lines.append(f"  runtime: {summary}")
    internet = report.get("internet")
    if internet:
        state = "OK" if internet.get("huggingface.co") else "NG (Settings -> Internet を On に)"
        lines.append(f"  huggingface.co        : {state}")
    for line in result["warnings"]:
        lines.append(f"  WARNING: {line}")
    if result["ok"]:
        lines.append("  -> unquantized pipeline can run here; run measure() for a real ETA.")
    lines.append("=================================")
    print("\n".join(lines), flush=True)
    return result


# ---------------------------------------------------------------------------
# Options and pipeline construction
# ---------------------------------------------------------------------------


@dataclass
class Options:
    """Everything one generation needs; safe to reuse across stages and sessions."""

    outdir: Path = Path("/kaggle/working/outputs/yue2-song")
    style: str = DEFAULT_STYLE
    lyrics: str = DEFAULT_LYRICS
    cot: str = "full"                    # full | melody | off
    seed: int = 831001
    seconds: float = 120.0               # audio length; frames = seconds / 0.04
    abc: str | None = None               # optional pre-written score (skips ABC planning)
    plan_max_tokens: int | None = None   # cap the ABC stage; None keeps the release 4096.
                                         # A truncated score is a quality change: it is for
                                         # smoke tests, not for comparable results.
    ode_steps: int = 32                  # release default; lower only as an explicit tradeoff
    model: str | None = None             # None -> /kaggle/input copy, else the HF repo
    vae: str | None = None
    revision: str | None = None
    vae_revision: str | None = None
    backend: str = "auto"                # auto | torch | torch-eager | vllm
    quantization: str = "none"
    offload_ar: bool | None = None       # None -> decided from VRAM
    vae_core_frames: int | None = None
    memory_budget_gib: float | None = None
    verify_hashes: bool = True
    token: str | None = None
    progress: bool = True
    budget_minutes: float = 600.0        # wall-clock guard for the whole run
    safety_minutes: float = 10.0         # stop this long before the budget is gone
    extra_semantic_margin: int = 64      # decode headroom beyond the requested frames
    frames: int = field(init=False)

    def __post_init__(self):
        self.outdir = Path(self.outdir)
        self.seconds = float(self.seconds)
        self.frames = frames_for_seconds(self.seconds)
        if self.cot not in {"full", "melody", "off"}:
            raise ValueError("cot must be full, melody or off")
        if self.abc is not None and self.cot == "off":
            raise ValueError("a supplied score requires cot=full or cot=melody")

    def semantic_sampling(self) -> dict:
        """Cap semantic tokens at the requested length (memory and runaway guard)."""
        budget = int(min(CONTEXT - 1024, self.frames + self.extra_semantic_margin))
        sampling = {"max_tokens": budget}
        if budget < 200:                     # release default min_tokens is 200
            sampling["min_tokens"] = max(1, budget - 8)
        return sampling

    def to_dict(self) -> dict:
        data = asdict(self)
        data["outdir"] = str(self.outdir)
        data["abc"] = None if self.abc is None else f"<{len(self.abc)} chars>"
        return data


def resolve_models(options: Options, report: dict | None = None) -> tuple[str, str]:
    model = options.model or (report or {}).get("local_model_dir") or DEFAULT_MODEL
    vae = options.vae or (report or {}).get("local_vae_dir") or DEFAULT_VAE
    return model, vae


def build_pipeline(report: dict, options: Options, backend: str, *, generation_config=None):
    """Construct a pipeline with device-appropriate memory settings."""
    from yue2 import YuE2Pipeline

    settings = pipeline_settings(report, ode_steps=options.ode_steps)
    model, vae = resolve_models(options, report)
    kwargs = dict(
        device="cuda",
        memory_budget_gib=options.memory_budget_gib or settings["memory_budget_gib"],
        backend=backend,
        quantization=options.quantization,
        offload_ar=settings["offload_ar"] if options.offload_ar is None else options.offload_ar,
        vae_core_frames=options.vae_core_frames or settings["vae_core_frames"],
        verify_hashes=options.verify_hashes,
        progress=options.progress,
    )
    if generation_config is not None:
        kwargs["generation_config"] = generation_config
    return YuE2Pipeline.from_pretrained(
        model, vae=vae, revision=options.revision, vae_revision=options.vae_revision,
        token=options.token, **kwargs)


def ode_config(options: Options):
    """Release generation config with the requested solver step count."""
    from yue2.protocol import GenerationConfig

    base = GenerationConfig().to_dict()
    base["ode_steps"] = int(options.ode_steps)
    return GenerationConfig.from_dict(base)


class PipelineHandle:
    """Pipeline plus the ability to rebuild when the GPU cannot execute the
    CUDA-graph/flash fast path (common on Turing/T4 and on old drivers)."""

    def __init__(self, report: dict, options: Options):
        self.report = report
        self.options = options
        self.generation_config = ode_config(options)
        self.backend = "torch" if options.backend == "auto" else options.backend
        self.pipe = build_pipeline(report, options, self.backend,
                                   generation_config=self.generation_config)

    def retry_eager(self, exc: BaseException) -> bool:
        """Rebuild with the eager backend after a graph/flash refusal."""
        message = str(exc).lower()
        if self.options.backend != "auto" or self.backend != "torch":
            return False
        if not any(hint in message for hint in GRAPH_FALLBACK_HINTS):
            return False
        print(f"[kaggle] {type(exc).__name__}: {exc}\n"
              "[kaggle] retrying with backend='torch-eager' (no CUDA graphs, no flash attention).",
              flush=True)
        self.close()
        self.backend = "torch-eager"
        self.pipe = build_pipeline(self.report, self.options, self.backend,
                                   generation_config=self.generation_config)
        return True

    def close(self):
        if getattr(self, "pipe", None) is not None:
            try:
                self.pipe.close()
            except Exception:                        # pragma: no cover - best effort
                pass
        self.pipe = None

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()


def run_with_retry(handle: PipelineHandle, stage):
    """Run ``stage(pipe)``; if this GPU refuses the graph path, rebuild and retry once."""
    try:
        return stage(handle.pipe)
    except (RuntimeError, NotImplementedError) as exc:
        if not handle.retry_eager(exc):
            raise
        return stage(handle.pipe)


def open_pipeline(report: dict, options: Options, *, log=print) -> PipelineHandle:
    handle = PipelineHandle(report, options)
    settings = pipeline_settings(report)
    log(f"[kaggle] pipeline ready: backend={handle.backend}, "
        f"device={handle.pipe.device}, offload_ar={handle.pipe.offload_ar}, "
        f"vae_core_frames={handle.pipe.vae_core_frames}, "
        f"budget={handle.pipe.memory_budget_gib:.1f} GiB "
        f"(device limit {settings['memory_budget_gib']:.1f} GiB)")
    return handle


# ---------------------------------------------------------------------------
# Honest estimates: measure the real model on the real GPU
# ---------------------------------------------------------------------------


def _timed(fn):
    start = time.perf_counter()
    value = fn()
    return value, time.perf_counter() - start


def measure(handle: PipelineHandle, *, ar_tokens: int = 16, frames: int = 128,
            log=print) -> dict:
    """Short real generations that bound this GPU's throughput.

    Costs a couple of minutes and answers "will my song finish in this session?"
    before anything long is started. Release sampling is used except for the
    hand-capped token budgets and the single solver step used for measurement.
    """
    import numpy as np
    from yue2.pipeline import SemanticResult
    from yue2.protocol import GenerationConfig

    options = handle.options
    out: dict = {"frames_measured": frames, "ar_tokens": ar_tokens}

    def plan_stage(pipe):                       # the measurement never uses the real ABC budget
        kwargs = dict(style=options.style, lyrics=options.lyrics, cot=options.cot,
                      seed=options.seed)
        if options.abc:
            kwargs["abc"] = options.abc
        if options.cot == "off":
            return pipe.plan(**kwargs)
        return pipe.plan(abc_sampling={"max_tokens": max(ar_tokens, 8), "min_tokens": 4}, **kwargs)

    plan, seconds = _timed(lambda: run_with_retry(handle, plan_stage))
    out["plan_seconds"] = seconds
    out["plan_external"] = plan.abc is not None and options.cot != "off"
    out["plan_tokens"] = len(plan.abc_ids)
    # output_tps excludes the prefill and is the number that matters for a long song.
    plan_tps = (plan.timing or {}).get("output_tps")
    if not plan_tps and out["plan_tokens"] and seconds > 0.25 and not out["plan_external"]:
        plan_tps = out["plan_tokens"] / seconds
    out["plan_tps"] = plan_tps
    log(f"[kaggle] planning probe: {seconds:.1f}s, {out['plan_tokens']} score tokens "
        f"(external score: {out['plan_external']})")

    semantic, seconds = _timed(lambda: run_with_retry(
        handle, lambda p: p.generate_semantic(plan, sampling={"max_tokens": max(ar_tokens, 8),
                                                              "min_tokens": 4})))
    out["semantic_seconds"] = seconds
    out["semantic_tokens"] = len(semantic.tokens)
    out["semantic_tps"] = (((semantic.timing or {}).get("output_tps"))
                           or ((len(semantic.tokens) / seconds) if seconds else None))
    log(f"[kaggle] semantic probe: {len(semantic.tokens)} tokens in {seconds:.1f}s "
        f"({out['semantic_tps']:.2f} tok/s)")

    # Flow matching: two sizes at one solver step give the linear fit
    #   time(n) = fixed_per_chunk + n * per_frame_per_velocity * (2 * steps)
    base = handle.generation_config
    try:
        handle.pipe.generation_config = GenerationConfig.from_dict({**base.to_dict(), "ode_steps": 1})
        rng = np.random.default_rng(options.seed)
        times = {}
        for size in (frames, frames * 2):
            tokens = rng.integers(0, 32768, size=size).tolist()
            probe = SemanticResult(plan, tokens, {}, False)
            _, seconds = _timed(lambda s=probe: run_with_retry(handle, lambda p: p.synthesize(s)))
            times[size] = seconds
            log(f"[kaggle] flow-matching probe: {size} frames in {seconds:.1f}s")
        slope = (times[frames * 2] - times[frames]) / frames     # 2 velocity evals per frame
        out["nf_per_frame_seconds"] = max(0.0, slope / 2)
        out["nf_fixed_per_chunk_seconds"] = max(0.0, times[frames] - slope * frames)
        out["ode_steps"] = int(base.ode_steps)
    finally:
        if handle.pipe is not None:
            handle.pipe.generation_config = handle.generation_config

    latents = np.random.default_rng(0).standard_normal((64, 64)).astype(np.float32)
    _, seconds = _timed(lambda: run_with_retry(handle, lambda p: p.decode(latents)))
    out["vae_seconds_per_frame"] = seconds / 64
    log(f"[kaggle] decoder probe: {seconds:.1f}s for 64 frames "
        f"({out['vae_seconds_per_frame'] * 1000:.0f} ms/frame)")
    return out


def predict(measured: dict, options: Options, *, plan=None, abc_tokens: int | None = None) -> dict:
    """Turn measurements into a per-stage ETA for the requested song."""
    frames = options.frames
    ode_steps = int(options.ode_steps)
    if plan is not None:
        prefix_tokens = len(plan.prefix)
    else:
        prefix_tokens = 64 + round(len(options.style) / 3.5) + round(len(options.lyrics) / 2.5) \
            + (round(len(options.abc) / 3.0) if options.abc else 0)
    chunk_size = max(1, (CONTEXT - prefix_tokens - 3) // 2)
    chunks = max(1, math.ceil(frames / chunk_size))

    stages: dict = {"frames": frames, "audio_seconds": seconds_for_frames(frames),
                    "chunks": chunks, "chunk_size": chunk_size}

    if options.cot == "off" or options.abc is not None:
        stages["plan"] = 0.0
    else:
        tokens = abc_tokens if abc_tokens is not None else min(4096, max(256, round(frames * 0.8)))
        stages["plan_tokens_assumed"] = tokens
        # Fall back to the semantic decode rate when the planning probe was never
        # truncated: both stages are the same AR decoder.
        tps = measured.get("plan_tps") or measured.get("semantic_tps")
        stages["plan_tps_used"] = tps
        stages["plan"] = (tokens / tps) if tps else None

    stages["semantic"] = (frames / measured["semantic_tps"]) if measured.get("semantic_tps") else None
    per_frame = measured.get("nf_per_frame_seconds")
    stages["flow_matching"] = (None if per_frame is None else
                               chunks * measured.get("nf_fixed_per_chunk_seconds", 0.0)
                               + frames * per_frame * 2 * ode_steps)
    stages["decode"] = measured.get("vae_seconds_per_frame", 0.0) * frames

    parts = [stages[key] for key in ("plan", "semantic", "flow_matching", "decode")]
    stages["total"] = sum(parts) if all(isinstance(v, float) for v in parts) else None
    usable = max(0.0, (options.budget_minutes - options.safety_minutes) * 60)
    stages["deadline_seconds"] = usable
    stages["fits_budget"] = stages["total"] is not None and stages["total"] <= usable
    return stages


def print_estimate(stages: dict, options: Options) -> None:
    print("--- estimated cost on this device ------------------------------------")
    print(f"  requested audio : {stages['audio_seconds']:.0f}s -> {stages['frames']} latent frames "
          f"(25 fps), {stages['chunks']} flow-matching chunk(s)")
    for key in ("plan", "semantic", "flow_matching", "decode", "total"):
        note = ""
        if key == "plan" and options.cot != "off" and options.abc is None:
            note = (f" (assumes {stages.get('plan_tokens_assumed')} score tokens at "
                    f"{stages.get('plan_tps_used')} tok/s; release cap 4096)")
        if key == "flow_matching":
            note = f" ({options.ode_steps} midpoint steps = {2 * int(options.ode_steps)} velocity evals)"
        print(f"  {key:<15}: {hms(stages.get(key))}{note}")
    if stages.get("total") is not None:
        print(f"  deadline        : {hms(stages['deadline_seconds'])} usable of "
              f"{options.budget_minutes:.0f} min  -> fits: {stages['fits_budget']}")
        if not stages["fits_budget"]:
            print("  NOTE: the estimate exceeds the wall-clock budget. Lower --seconds, lower "
                  "ode_steps, or split the run across sessions (the stages resume).")
    print("---------------------------------------------------------------------", flush=True)


# ---------------------------------------------------------------------------
# Staged generation with resume
# ---------------------------------------------------------------------------


def _load_semantic(plan, path: Path, metadata: dict):
    """Rebuild a SemanticResult from cached tokens."""
    import numpy as np
    from yue2.pipeline import SemanticResult

    tokens = np.load(path, allow_pickle=False)
    if tokens.ndim != 1 or tokens.dtype.kind not in "iu":
        raise ValueError(f"unexpected cached token array in {path}")
    return SemanticResult(plan, [int(t) for t in tokens.tolist()],
                          metadata.get("timing", {}), bool(metadata.get("truncated", False)))


def _synthesize_checkpointed(pipe, semantic, outdir: Path, *, log=print, cancelled=None):
    """``YuE2Pipeline.synthesize`` with one cached file per original chunk.

    Calls the module-level helpers the pipeline itself calls (``yue2.nar``) so a
    long song survives a session timeout; falls back to the public method if the
    internal signature ever changes.
    """
    import numpy as np

    outdir.mkdir(parents=True, exist_ok=True)
    try:
        from yue2.nar import CachedNAR, song_chunks
        config = pipe.generation_config
        model = pipe._load_model(for_nar=True)          # the same call synthesize() makes
        chunks = song_chunks(semantic.plan.prefix, semantic.tokens,
                             semantic.plan.request.seed, config.context)
    except (ImportError, AttributeError, TypeError) as exc:
        log(f"[kaggle] checkpointed synthesis unavailable ({type(exc).__name__}: {exc}); "
            "using public synthesize() without per-chunk resume")
        return pipe.synthesize(semantic, cancelled=cancelled)

    parts = []
    for index, chunk in enumerate(chunks):
        path = outdir / f"latent_chunk_{index:04d}.npy"
        if path.is_file():
            parts.append(np.load(path))
            log(f"[kaggle] chunk {index + 1}/{len(chunks)} resumed from disk")
            continue
        if cancelled is not None and cancelled():
            raise InterruptedError("cancelled before a flow-matching chunk")
        engine = CachedNAR(model, chunk)
        try:
            state = {"done": 0}

            def progress(completed, total, state=state, index=index):
                if completed - state["done"] >= max(1, total // 10):
                    state["done"] = completed
                    log(f"[kaggle] chunk {index + 1}/{len(chunks)}: {completed}/{total} steps")

            latents = engine.solve(steps=int(pipe.generation_config.ode_steps),
                                   cancelled=cancelled, on_progress=progress)
        finally:
            engine.close()
        array = np.asarray(latents, dtype=np.float32)
        np.save(path, array)
        parts.append(array)
        log(f"[kaggle] chunk {index + 1}/{len(chunks)} done")
    return np.concatenate(parts, axis=0) if len(parts) > 1 else parts[0]


def _stage_plan(pipe, options: Options, *, log=print, cancelled=None):
    """``plan()`` with the caller's request.

    Release sampling is untouched unless the caller explicitly capped the ABC
    budget with ``plan_max_tokens``.
    """
    kwargs = dict(style=options.style, lyrics=options.lyrics, cot=options.cot,
                  seed=options.seed, cancelled=cancelled)
    if options.abc:
        kwargs["abc"] = options.abc
    if options.plan_max_tokens and options.cot != "off":
        kwargs["abc_sampling"] = {"max_tokens": int(options.plan_max_tokens),
                                  "min_tokens": min(32, int(options.plan_max_tokens))}
    return pipe.plan(**kwargs)


def run(handle: PipelineHandle, options: Options, *, rates: dict | None = None,
        resume: bool = True, log=print) -> dict:
    """Execute the four stages, caching each one inside ``options.outdir``."""
    import numpy as np
    from yue2 import SymbolicPlan
    from yue2.pipeline import SongResult
    from yue2.storage import identity, write_json

    outdir = options.outdir
    outdir.mkdir(parents=True, exist_ok=True)
    state_path = outdir / "kaggle_run.json"
    deadline = time.time() + max(60.0, (options.budget_minutes - options.safety_minutes) * 60)
    cancelled = lambda: time.time() > deadline          # noqa: E731 - wall-clock guard

    state: dict = {"options": options.to_dict(), "started": time.strftime("%Y-%m-%d %H:%M:%S"),
                   "budget_minutes": options.budget_minutes, "stages": {}}
    if state_path.is_file():
        try:
            state["previous"] = json.loads(state_path.read_text(encoding="utf-8")).get("stages")
        except ValueError:
            pass
    if rates:
        state["measured"] = rates

    def note(stage, seconds, extra=None):
        entry = {"seconds": round(seconds, 2), "at": time.strftime("%H:%M:%S")}
        entry.update(extra or {})
        state["stages"][stage] = entry
        write_json(state_path, state)
        log(f"[kaggle] stage '{stage}' finished in {hms(seconds)}")

    result_path = outdir / "result.json"
    if resume and result_path.is_file():
        log(f"[kaggle] already complete: {result_path}")
        return {"status": "already-complete", "outdir": str(outdir)}

    pipe = handle.pipe

    def guarded(label, call):
        """Run a stage; a wall-clock stop is reported instead of raised so the
        notebook keeps the cached stages and the session stays usable."""
        try:
            return call()
        except InterruptedError as exc:
            state["interrupted"] = {"stage": label, "reason": str(exc),
                                    "at": time.strftime("%H:%M:%S")}
            write_json(state_path, state)
            log(f"[kaggle] wall-clock guard stopped '{label}' ({exc}); "
                "re-run to continue from the cached stages.")
            return None

    # ---- stage 1: symbolic plan -------------------------------------------------
    if resume and (outdir / "plan_manifest.json").is_file():
        plan = SymbolicPlan.load(outdir)
        log(f"[kaggle] stage 'plan' resumed from disk ({len(plan.abc_ids)} score tokens)")
    else:
        outcome = guarded("plan", lambda: _timed(lambda: run_with_retry(
            handle, lambda p: _stage_plan(p, options, log=log, cancelled=cancelled))))
        if outcome is None:
            return {"status": "interrupted", "stage": "plan", "outdir": str(outdir)}
        plan, seconds = outcome
        plan.save(outdir)
        note("plan", seconds, {"truncated": bool(plan.truncated), "score_tokens": len(plan.abc_ids)})
        if plan.truncated:
            log("[kaggle] WARNING: planning hit its token cap, so the score is incomplete. "
                "Shorten the song or supply your own ABC if the result sounds clipped.")

    # ---- stage 2: semantic tokens ----------------------------------------------
    semantic_file = outdir / "semantic.npy"
    if resume and semantic_file.is_file():
        semantic = _load_semantic(plan, semantic_file, state["stages"].get("semantic") or {})
        log(f"[kaggle] stage 'semantic' resumed from disk ({len(semantic.tokens)} tokens)")
    else:
        outcome = guarded("semantic", lambda: _timed(lambda: run_with_retry(
            handle, lambda p: p.generate_semantic(plan, sampling=options.semantic_sampling(),
                                                  cancelled=cancelled))))
        if outcome is None:
            return {"status": "interrupted", "stage": "semantic", "outdir": str(outdir)}
        semantic, seconds = outcome
        np.save(semantic_file, np.asarray(semantic.tokens, dtype=np.int32))
        note("semantic", seconds, {"tokens": len(semantic.tokens),
                                   "truncated": bool(semantic.truncated)})
        if semantic.truncated:
            log("[kaggle] WARNING: semantic generation hit its token cap; the song may end early.")

    # ---- stage 3: flow matching ------------------------------------------------
    latent_file = outdir / "latent.npy"
    if resume and latent_file.is_file():
        latents = np.load(latent_file)
        log(f"[kaggle] stage 'flow_matching' resumed from disk ({latents.shape[0]} frames)")
    else:
        outcome = guarded("flow_matching", lambda: _timed(lambda: run_with_retry(
            handle, lambda p: _synthesize_checkpointed(p, semantic, outdir, log=log,
                                                       cancelled=cancelled))))
        if outcome is None:
            return {"status": "interrupted", "stage": "flow_matching", "outdir": str(outdir)}
        latents, seconds = outcome
        latents = np.asarray(latents, dtype=np.float32)
        np.save(latent_file, latents)
        note("flow_matching", seconds, {"frames": int(latents.shape[0])})

    # ---- stage 4: decode and canonical artifacts -------------------------------
    if resume and (outdir / "audio.flac").is_file() and result_path.is_file():
        log("[kaggle] stage 'decode' already complete")
        return {"status": "already-complete", "outdir": str(outdir)}

    audio, seconds = _timed(lambda: run_with_retry(handle, lambda p: p.decode(latents)))
    request = semantic.plan.request
    config = pipe.effective_config(request, None, options.semantic_sampling())
    song = SongResult(audio, 48000, semantic, latents, config, pipe.weights,
                      {"kaggle_stages": state["stages"]},
                      identity({"request": request.to_dict(), "config": config,
                                "weights": pipe.weights}))
    # result.json records a hash for every file in the directory, so keep the
    # runner's own log out of the directory while that manifest is written.
    if state_path.is_file():
        state_path.unlink()
    try:
        receipt = song.save_artifacts(outdir)
    except BaseException:
        write_json(state_path, state)
        raise
    state["stages"]["decode"] = {"seconds": round(seconds, 2), "at": time.strftime("%H:%M:%S"),
                                 "audio_seconds": receipt.get("audio_seconds")}
    state["completed"] = time.strftime("%Y-%m-%d %H:%M:%S")
    state["truncated"] = song.truncated
    write_json(state_path, state)
    log(f"[kaggle] stage 'decode' finished in {hms(seconds)}")
    log(f"[kaggle] done -> {outdir / 'audio.flac'} ({receipt.get('audio_seconds', 0):.1f}s of audio)")
    return {"status": "complete", "outdir": str(outdir), "truncated": song.truncated,
            "receipt": receipt}


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------


def _add_common(parser):
    parser.add_argument("--outdir", type=Path, default=Path("/kaggle/working/outputs/yue2-song"))
    parser.add_argument("--style", default=DEFAULT_STYLE)
    parser.add_argument("--lyrics", default=None)
    parser.add_argument("--lyrics-file", type=Path, default=None)
    parser.add_argument("--abc-file", type=Path, default=None)
    parser.add_argument("--cot", choices=("full", "melody", "off"), default="full")
    parser.add_argument("--seed", type=int, default=831001)
    parser.add_argument("--seconds", type=float, default=120.0)
    parser.add_argument("--ode-steps", type=int, default=32)
    parser.add_argument("--plan-max-tokens", type=int, default=None,
                        help="cap the ABC planning stage (smoke tests only; truncates the score)")
    parser.add_argument("--model", default=None)
    parser.add_argument("--vae", default=None)
    parser.add_argument("--revision", default=None)
    parser.add_argument("--vae-revision", default=None)
    parser.add_argument("--backend", choices=("auto", "torch", "torch-eager", "vllm"), default="auto")
    parser.add_argument("--offload-ar", dest="offload_ar", action="store_true", default=None)
    parser.add_argument("--no-offload-ar", dest="offload_ar", action="store_false")
    parser.add_argument("--budget-minutes", type=float, default=600.0)
    parser.add_argument("--safety-minutes", type=float, default=10.0)
    parser.add_argument("--quiet", action="store_true")


def _options(args) -> Options:
    lyrics = args.lyrics
    if args.lyrics_file:
        lyrics = args.lyrics_file.read_bytes().decode("utf-8")
    options = Options(outdir=args.outdir, style=args.style,
                      lyrics=lyrics if lyrics is not None else DEFAULT_LYRICS,
                      cot=args.cot, seed=args.seed, seconds=args.seconds,
                      ode_steps=args.ode_steps, plan_max_tokens=args.plan_max_tokens,
                      model=args.model, vae=args.vae,
                      revision=args.revision, vae_revision=args.vae_revision,
                      backend=args.backend, offload_ar=args.offload_ar,
                      budget_minutes=args.budget_minutes,
                      safety_minutes=args.safety_minutes, progress=not args.quiet)
    if args.abc_file:
        options.abc = args.abc_file.read_bytes().decode("utf-8")
    return options


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    sub = parser.add_subparsers(dest="command", required=True)
    check = sub.add_parser("check", help="preflight only: device, memory, verdict")
    check.add_argument("--json", type=Path, default=None)
    bench = sub.add_parser("bench", help="preflight, real micro-benchmark, ETA")
    _add_common(bench)
    run_cmd = sub.add_parser("run", help="staged, resumable song generation")
    _add_common(run_cmd)
    run_cmd.add_argument("--skip-bench", action="store_true")
    args = parser.parse_args(argv)

    try:
        report = environment()
    except ImportError as exc:
        print(f"torch is not importable in this kernel ({exc}).\n"
              "Run the install cell first (Kaggle images already ship a CUDA torch).")
        return 2
    if not print_report(report)["ok"]:
        return 2
    if args.command == "check":
        if getattr(args, "json", None):
            args.json.parent.mkdir(parents=True, exist_ok=True)
            args.json.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
        return 0

    options = _options(args)
    with open_pipeline(report, options) as handle:
        rates = None
        if args.command == "bench" or not args.skip_bench:
            try:
                rates = measure(handle)
                print_estimate(predict(rates, options), options)
            except Exception as exc:                 # pragma: no cover - device dependent
                print(f"[kaggle] benchmark failed ({type(exc).__name__}: {exc}); continuing")
        if args.command == "bench":
            return 0
        outcome = run(handle, options, rates=rates)
        print(json.dumps(outcome, indent=2, ensure_ascii=False, default=str)[:4000])
        return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")          # the %%writefile cell above
sys.path.append(str(REPO_DIR / "kaggle"))      # offline/local fallback: use the repo copy
import importlib, yue2_kaggle as yk
importlib.reload(yk)
print("driver:", yk.__file__)

## 5. プリフライト

In [ ]:
# --- プリフライト: デバイス・VRAM・ディスク・BF16 判定（pipeline と同じ判定を使う） ---
report = yk.environment()
READY = yk.print_report(report)["ok"]

if not READY:
    print("\n>>> この環境では実行できません。Settings で Accelerator を 'GPU T4 x2' に変えて再実行してください。")
else:
    settings = yk.pipeline_settings(report)
    print("recommended:", {k: round(v, 2) if isinstance(v, float) else v for k, v in settings.items()})

## 6. 実行オプション

In [ ]:
# --- 実行オプションを組み立てる（設定セル＋環境から） ---

# HF トークン（モデルが gated の場合は Kaggle Secrets に HF_TOKEN を登録）
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

opts = yk.Options(
    outdir=OUTDIR, style=STYLE, lyrics=LYRICS, cot=COT, seed=SEED, seconds=SECONDS,
    ode_steps=ODE_STEPS, abc=ABC, token=HF_TOKEN, plan_max_tokens=PLAN_MAX_TOKENS,
    budget_minutes=BUDGET_MINUTES, safety_minutes=SAFETY_MINUTES,
)
print("曲の長さ:", opts.frames, "frames =", f"{opts.seconds:.0f}s",
      "| semantic max_tokens:", opts.semantic_sampling()["max_tokens"])
print("モデル:", yk.resolve_models(opts, report))
print("既存の出力:", sorted(p.name for p in OUTDIR.iterdir()) if OUTDIR.exists() else "(なし=新規)")

## 7. 実測ベンチ → 生成（再開可能）

In [ ]:
# --- ベンチ → ETA → 4段ステージ実行（途中で落ちても再実行で再開） ---
if not READY:
    raise RuntimeError("プリフライトに失敗しています。上のセルを確認してください。")

with yk.open_pipeline(report, opts) as handle:
    rates = None
    if RUN_BENCHMARK:
        rates = yk.measure(handle)
        stages = yk.predict(rates, opts)
        yk.print_estimate(stages, opts)
        if not stages["fits_budget"] and not FORCE_RUN:
            print("\n>>> ETA が予算を超えています。SECONDS を下げる / ODE_STEPS を下げる / "
                  "FORCE_RUN=True で強行、のいずれかを選んでください。")
            outcome = {"status": "skipped", "reason": "estimate exceeds budget"}
        else:
            outcome = yk.run(handle, opts, rates=rates)
    else:
        outcome = yk.run(handle, opts)

print(outcome.get("status"), "->", outcome.get("outdir"))
if outcome.get("status") == "interrupted":
    print("時間切れで安全に停止しました（完了した段は保存済み）。"
          "そのまま再実行すると続きから再開します。")

## 8. 結果

In [ ]:
# --- 結果の確認と試聴 ---
import json
from pathlib import Path
from IPython.display import Audio, display

audio_path = OUTDIR / "audio.flac"
if audio_path.is_file():
    result = json.loads((OUTDIR / "result.json").read_text(encoding="utf-8"))
    print(f"audio: {result['audio_seconds']:.1f}s @ {result['sample_rate']} Hz "
          f"| truncated: {result['truncated']}")
    print("files:", ", ".join(f"{p.name} ({p.stat().st_size/1e6:.1f} MB)"
                              for p in sorted(OUTDIR.iterdir())))
    display(Audio(filename=str(audio_path)))
    print("\n曲の構成(ABC 譜)を確認するには score.abc を開いてください:",
          OUTDIR / "score.abc" if (OUTDIR / "score.abc").exists() else "(ABC なし: cot=off)")
    run_state = OUTDIR / "kaggle_run.json"
    if run_state.is_file():
        print(json.dumps(json.loads(run_state.read_text(encoding='utf-8')).get("stages", {}),
                         indent=1, ensure_ascii=False))
else:
    print("まだ音声がありません。上の実行セルを走らせてください（再実行で続きから再開します）。")

## 詰まりやすい点とコツ

**時間の見積もり。** 公式リポジトリの資料やコミュニティ計測では、RTX 4090 で 3.6 分の曲に約 71 秒
とされています。T4 は BF16 がネイティブでなくメモリ帯域も 1/3 程度なので、**同条件で 10〜30 倍程度**かかる
ことを想定してください。正確な数字はこのノートの実測ベンチが出します。内訳の支配項は
flow matching（`frames × 28層 × 64 velocity evals`）で、`ODE_STEPS` と曲の長さにほぼ比例します。

**短く試す。** まず `SECONDS=60〜120`, `COT="melody"` + 手持ち ABC, `ODE_STEPS=8〜16` で
「鳴るかどうか」を確かめ、それから本番設定に上げるのが安全です。`cot="off"` は譜面生成を省ける代わりに
guidance 1.01（CFG 2分岐）になり、メモリも計算もほぼ倍になるので T4 では不利です。

**時間切れの扱い。** 実行セルは `BUDGET_MINUTES - SAFETY_MINUTES` を過ぎると pipeline の
`cancelled` フックで**安全に停止**します（例外で落ちず、status が `interrupted` になります）。
このときも完了済みの段はディスクに残っているので、再実行で続きから進みます。

**再開の仕組み。** 出力ディレクトリに `plan_manifest.json` / `semantic.npy` / `latent.npy` /
`latent_chunk_XXXX.npy` / `audio.flac` が段ごとに保存されます。同じ `OUTPUT_ID` で再実行すると、
終わった段はスキップして続きから進みます。`Plan` が保存されているので、歌詞を変えずに再開するのは安全です
（歌詞やスタイルを変えるときは `OUTPUT_ID` を変えてください）。

**セッションを跨ぐコスト。** モデル(約 7.8 GB)はセッションごとに Hugging Face から取り直します
（数分）。`PERSIST_MODEL=True` にすると `/kaggle/working/hf-cache` に置かれ、そのフォルダを
Kaggle Dataset（Private）にすれば次回 `/kaggle/input` から自動検出して再利用できます。

**メモリ。** このノートは `memory_budget_gib` を VRAM から自動決定し、`vae_core_frames=512`、
必要なら `offload_ar=True` を使います。それでも OOM になる場合は `SECONDS` を短くしてください
（semantic の KV キャッシュは要求長に比例して縮みます）。

**品質についての注意。** T4 の BF16 はエミュレーションで**数値は正しくても速くない**だけです。
一方 `ODE_STEPS` を下げる/譜面なし(`cot="off"`)にするといった変更は品質そのものを変えます。
比べるときは同じ条件どうしで比べ、結果を「公式ベンチマークの再現」と呼ばないでください。

**うまくいかないとき。**
- `FlashAttention only supports Ampere GPUs` などが出たら、このノートは自動で `torch-eager` に切り替えます
  （CUDA graph / flash を使わない経路。遅いが T4 でも動きます）。
- `ImportError: cannot import name '_center' from 'numpy._core.umath'` → numpy を差し替えた影響です。
  Run → Restart session して `INSTALL_MODE="image"`（既定）で上から実行し直してください。
- GPU が P100 だった → Settings で `GPU T4 x2` に変更。
- CUDA OOM → `SECONDS` を下げる、`ODE_STEPS` を下げる、ノートを再起動。
- 生成が途中で止まった → 出力ディレクトリを確認し、そのまま再実行（続きから再開）。

**代替手段。** どうしても T4 で時間が足りない場合、量子化版（GGUF / MLX）や
公式のオンラインデモ・API を使う手もあります。このリポジトリの公式要件（24 GB BF16）を満たす
GPU が使えるなら、そちらの方が確実です。

## 参考

- 公式リポジトリ: https://github.com/multimodal-art-projection/YuE
- モデル: https://huggingface.co/m-a-p/YuE2-3B / https://huggingface.co/m-a-p/YuE2-Vae
- ライセンス: 重みは CC BY-NC 4.0（個人の制作物の収益化は許諾、企業の商用は要相談）